In [1]:
!pip install rasterio

In [2]:
import rasterio
import numpy as np

In [3]:
import rasterio

b02_path = "/content/B02.tiff"
b03_path = "/content/B03.tiff"
b04_path = "/content/B04.tiff"
b08_path = "/content/B08.tiff"

with rasterio.open(b02_path) as src:
    print("B02:", src.count, src.crs)

with rasterio.open(b03_path) as src:
    print("B03:", src.count, src.crs)

with rasterio.open(b04_path) as src:
    print("B04:", src.count, src.crs)

with rasterio.open(b08_path) as src:
    print("B08:", src.count, src.crs)

B02: 1 EPSG:4326
B03: 1 EPSG:4326
B04: 1 EPSG:4326
B08: 1 EPSG:4326


In [4]:
with rasterio.open(b02_path) as src:
    B02 = src.read(1)

with rasterio.open(b03_path) as src:
    B03 = src.read(1)

with rasterio.open(b04_path) as src:
    B04 = src.read(1)

with rasterio.open(b08_path) as src:
    B08 = src.read(1)

print("B03 shape:", B03.shape)
print("B04 shape:", B04.shape)
print("B08 shape:", B08.shape)

B03 shape: (1188, 2500)
B04 shape: (1188, 2500)
B08 shape: (1188, 2500)


In [5]:
ndwi = np.full(B03.shape, np.nan, dtype="float32")
ndti = np.full(B04.shape, np.nan, dtype="float32")

valid_1 = (B03 + B04) != 0
valid = (B03 + B08) != 0

ndwi[valid] = (B03[valid] - B08[valid]) / (B03[valid] + B08[valid])
ndti[valid_1] = (B04[valid_1] - B03[valid_1]) / (B04[valid_1] + B03[valid_1])


print("NDWI shape:", ndwi.shape)
print("NDWI min:", np.nanmin(ndwi))
print("NDWI max:", np.nanmax(ndwi))

print("\nNDTI shape:", ndti.shape)
print("NDTI min:", np.nanmin(ndti))
print("NDTI max:", np.nanmax(ndti))

NDWI shape: (1188, 2500)
NDWI min: -1.0
NDWI max: 1.0

NDTI shape: (1188, 2500)
NDTI min: -1.0
NDTI max: 1.0


In [6]:
stacked_data = np.stack([B02, B03, B04, B08, ndwi, ndti], axis=0)

print("Stacked shape:", stacked_data.shape)

Stacked shape: (6, 1188, 2500)


In [7]:
patch = stacked_data[:, :32, :32]

print('Patch shape:', patch.shape)

Patch shape: (6, 32, 32)


In [14]:
patches = []
coords = []


for row in range(0, 1188 - 32 + 1, 32):
    for col in range(0, 2500 - 32 + 1, 32):

        patch = stacked_data[:, row:row+32, col:col+32]

        patches.append(patch)
        coords.append((row, col))

print("Total patches:", len(patches))

print("Patch 1 shape:", patches[0].shape)
print("Patch 2 shape:", patches[1].shape)
print("Patch 3 shape:", patches[2].shape)

Total patches: 2886
Patch 1 shape: (6, 32, 32)
Patch 2 shape: (6, 32, 32)
Patch 3 shape: (6, 32, 32)


In [9]:
import torch
from torch.utils.data import Dataset

In [23]:
from rasterio.transform import xy

def pixel_to_latlon(transform, row, col):
    lon, lat = xy(transform, row, col, offset="center")
    return lat, lon

In [24]:
import math

def haversine_km(lat1, lon1, lat2, lon2):
    R = 6371.0

    lat1 = math.radians(lat1)
    lat2 = math.radians(lat2)

    dlat = lat2 - lat1
    dlon = math.radians(lon2 - lon1)

    a = (
        math.sin(dlat / 2) ** 2
        + math.cos(lat1)
        * math.cos(lat2)
        * math.sin(dlon / 2) ** 2
    )

    c = 2 * math.atan2(math.sqrt(a), math.sqrt(1 - a))

    return R * c


def nearest_sensor(lat, lon, sensors):
    best_sensor = None
    best_distance = float("inf")

    for sensor in sensors:
        distance = haversine_km(
            lat,
            lon,
            sensor["lat"],
            sensor["lon"]
        )

        if distance < best_distance:
            best_distance = distance
            best_sensor = sensor

    return best_sensor

In [10]:
class FusionDataset(Dataset):

    def __init__(self, patches, coords, transform, sensors):
        self.patches = patches
        self.coords = coords
        self.transform = transform
        self.sensors = sensors

    def __len__(self):
        return len(self.patches)

    def __getitem__(self, index):

        patch = self.patches[index]

        row, col = self.coords[index]

        center_row = row + 16
        center_col = col + 16

        lat, lon = pixel_to_latlon(
            self.transform,
            center_row,
            center_col
        )

        sensor = nearest_sensor(
            lat,
            lon,
            self.sensors
        )

        ntu = sensor["ntu"]

        patch = torch.tensor(
            patch,
            dtype=torch.float32
        )

        ntu = torch.tensor(
            ntu,
            dtype=torch.float32
        )

        return patch, ntu

In [25]:
import rasterio

with rasterio.open("/content/Jul_polluted.tif") as src:
    transform = src.transform

In [18]:
import csv

sensors = []

with open("/content/Jul_sensors.csv", newline="", encoding="utf-8") as f:
    for rec in csv.DictReader(f):
        sensors.append({
            "station_id": rec["Station_ID"],
            "lat": float(rec["Lat"]),
            "lon": float(rec["Lon"]),
            "ntu": float(rec["Turbidity_NTU"])
        })

In [26]:
dataset = FusionDataset(
    patches,
    coords,
    transform,
    sensors
)

print("Number of samples:", len(dataset))

patch, ntu = dataset[0]

print("Patch type:", type(patch))
print("Patch shape:", patch.shape)

print("NTU type:", type(ntu))
print("NTU value:", ntu)

Number of samples: 2886
Patch type: <class 'torch.Tensor'>
Patch shape: torch.Size([6, 32, 32])
NTU type: <class 'torch.Tensor'>
NTU value: tensor(5.)
